**Nama:** Muhammad Adam Hasani
**NIM:** 25200013
**Mata Kuliah:** Data Wrangling

# Tugas 1 - Integrasi Data Multi-Sumber

Studi kasus PT Nusantara Retail Mandiri. Data transaksi ada di CSV, data pelanggan di database SQL, status pengiriman ambil dari API logistik.

## Bagian A - Analisis Konseptual & Tata Kelola Data

### 1. Pemetaan Kebutuhan Bisnis ke Kebutuhan Teknis

Dari pertanyaan Data Owner, saya breakdown jadi atribut yang perlu diambil:

| Atribut | Tipe Data | Sumber |
|---|---|---|
| InvoiceNo | STRING | CSV POS |
| InvoiceDate | DATETIME | CSV POS |
| Quantity & UnitPrice | INTEGER / FLOAT | CSV POS |
| total_amount | FLOAT | CSV POS (Quantity x UnitPrice) |
| coupon_code | STRING | CSV POS |
| CustomerID | INTEGER | CSV POS & Database CRM |
| membership_tier | VARCHAR | Database CRM |
| loyalty_points | INTEGER | Database CRM |
| is_active | INTEGER (0/1) | Database CRM |
| status pengiriman | STRING | REST API logistik |
| courier | STRING | REST API logistik |
| tracking_number | STRING | REST API logistik |

### 2. Matriks RACI

Keterangan: R = Responsible, A = Accountable, C = Consulted, I = Informed

| Aktivitas | Head of Business Strategy | Data Management Office | Tim Data Wrangling | BI Analyst |
|---|---|---|---|---|
| Menentukan kebutuhan informasi bisnis | A | C | I | C |
| Menetapkan standar data | I | A | C | I |
| Ekstraksi & integrasi data multi-sumber | I | C | R | I |
| Validasi kualitas data | I | A | R | I |
| Pembuatan kamus data | I | A | R | C |
| Analisis performa kampanye kupon | A | I | C | R |

## Bagian B.1 - Ekstraksi CSV (Divisi Kasir/POS)

In [1]:
import pandas as pd
import numpy as np

df_pos = pd.read_csv(
    'transaksi_pos_nrm.csv',
    parse_dates=['InvoiceDate'],
    dtype={'InvoiceNo': str, 'StockCode': str}
)

In [2]:
df_pos.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,total_amount,coupon_code,coupon_discount
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30,NaN,NaN
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,NaN,NaN
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00,NaN,NaN
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,NaN,NaN
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,NaN,NaN


In [3]:
df_pos.info()

print('jumlah baris :', len(df_pos))
print('jumlah invoice:', df_pos['InvoiceNo'].nunique())
print('jumlah customer:', df_pos['CustomerID'].nunique())

<class 'pandas.DataFrame'>
RangeIndex: 80000 entries, 0 to 79999
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   InvoiceNo        80000 non-null  str           
 1   StockCode        80000 non-null  str           
 2   Description      80000 non-null  str           
 3   Quantity         80000 non-null  int64         
 4   InvoiceDate      80000 non-null  datetime64[us]
 5   UnitPrice        80000 non-null  float64       
 6   CustomerID       80000 non-null  int64         
 7   Country          80000 non-null  str           
 8   total_amount     80000 non-null  float64       
 9   coupon_code      4171 non-null   str           
 10  coupon_discount  4171 non-null   float64       
dtypes: datetime64[us](1), float64(3), int64(2), str(5)
memory usage: 6.7 MB
jumlah baris : 80000
jumlah invoice: 4005
jumlah customer: 1904


## Bagian B.2 - Koneksi Database (Divisi CRM/Membership)

Ambil pelanggan aktif saja.

In [4]:
from sqlalchemy import create_engine

engine = create_engine('sqlite:///customers_nrm.db')

df_crm = pd.read_sql("""SELECT customer_id, email, gender, age, city,
                           membership_tier, loyalty_points, is_active
                    FROM customers
                    WHERE is_active = 1""", engine)

df_crm.head()

,customer_id,email,gender,age,city,membership_tier,loyalty_points,is_active
0,12346,customer12346@mail.com,L,19,Yogyakarta,Gold,2006,1
1,12347,customer12347@mail.com,L,61,Yogyakarta,Bronze,4837,1
2,12348,customer12348@mail.com,L,23,Bandung,Silver,4139,1
3,12350,customer12350@mail.com,L,63,Yogyakarta,Platinum,1805,1
4,12352,customer12352@mail.com,P,18,Semarang,Silver,3462,1


In [5]:
print('pelanggan aktif:', len(df_crm), 'dari total 1904')
df_crm['membership_tier'].value_counts()

pelanggan aktif: 1755 dari total 1904


membership_tier
Platinum    458
Silver      450
Gold        434
Bronze      413
Name: count, dtype: int64

## Bagian B.3 - Konsumsi REST API (Mitra Logistik)

Endpoint logistik nggak publik, jadi saya mock jadi file JSON yang saya taruh di GitHub. JSON-nya bersarang, diratakan pakai pd.json_normalize.

In [6]:
import requests, json

url = 'https://raw.githubusercontent.com/adamhasani/tugas1-data-wrangling/master/status_pengiriman.json'

try:
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    api_data = resp.json()
    print('ambil data dari API, status:', resp.status_code)
except Exception as e:
    print('gagal ambil dari API:', e)
    print('pakai file lokal saja')
    with open('status_pengiriman.json') as f:
        api_data = json.load(f)

ambil data dari API, status: 200


In [7]:
# lihat struktur JSON aslinya dulu
print(json.dumps(api_data['data'][0], indent=2))

{
  "order_id": "536365",
  "shipment": {
    "tracking_number": "NRM37C9CE5E5E",
    "courier": "AnterAja",
    "service": "YES"
  },
  "status": "Delivered",
  "timeline": {
    "shipped_at": "2011-02-13 00:00:00",
    "delivered_at": "2011-02-16 00:00:00"
  },
  "customer_id": 17850
}


In [8]:
df_logistik = pd.json_normalize(api_data['data'], sep='_')
print(df_logistik.shape)
df_logistik.head()

(4005, 8)


,order_id,status,customer_id,shipment_tracking_number,shipment_courier,shipment_service,timeline_shipped_at,timeline_delivered_at
0,536365,Delivered,17850,NRM37C9CE5E5E,AnterAja,YES,2011-02-13 00:00:00,2011-02-16 00:00:00
1,536366,Delivered,17850,NRMF197DB58BD,SiCepat,SameDay,2011-02-28 00:00:00,2011-03-05 00:00:00
2,536367,Shipped,13047,NRM7BA21B6ED2,SiCepat,Economy,2011-01-27 00:00:00,NaN
3,536368,Delivered,13047,NRM5F5642120F,Ninja,Economy,2011-01-13 00:00:00,2011-01-20 00:00:00
4,536369,Delivered,13047,NRMFCBB6C31D2,AnterAja,Economy,2011-02-09 00:00:00,2011-02-12 00:00:00


## Bagian B.4 - Integrasi Data (Merging)

Ketiganya digabung pakai LEFT JOIN: transaksi tanpa record pengiriman dan pelanggan non-aktif tetap ikut dihitung. Logistik dan POS di-dedup dulu biar join nggak menggandakan baris.

In [9]:
# cek dulu ada berapa baris duplikat
print('duplikat logistik:', df_logistik.duplicated(subset='order_id').sum())
print('duplikat POS:', df_pos.duplicated(subset=['InvoiceNo', 'StockCode']).sum())

duplikat logistik: 0
duplikat POS: 1965


In [10]:
df_logistik = df_logistik.drop_duplicates(subset='order_id')
df_pos = df_pos.drop_duplicates(subset=['InvoiceNo', 'StockCode'])

# gabung POS + logistik
df_master_analisis = df_pos.merge(
    df_logistik,
    how='left',
    left_on='InvoiceNo',
    right_on='order_id'
)

# tambah data CRM
df_master_analisis = df_master_analisis.merge(
    df_crm,
    how='left',
    on='customer_id'
)

df_master_analisis = df_master_analisis.drop(columns=['order_id'])

print('hasil akhir:', df_master_analisis.shape)
df_master_analisis.head()

hasil akhir: (78035, 25)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,total_amount,coupon_code,...,shipment_service,timeline_shipped_at,timeline_delivered_at,email,gender,age,city,membership_tier,loyalty_points,is_active
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30,NaN,...,YES,2011-02-13 00:00:00,2011-02-16 00:00:00,customer17850@mail.com,L,37.0,Jakarta,Gold,2313.0,1.0
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,NaN,...,YES,2011-02-13 00:00:00,2011-02-16 00:00:00,customer17850@mail.com,L,37.0,Jakarta,Gold,2313.0,1.0
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00,NaN,...,YES,2011-02-13 00:00:00,2011-02-16 00:00:00,customer17850@mail.com,L,37.0,Jakarta,Gold,2313.0,1.0
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,NaN,...,YES,2011-02-13 00:00:00,2011-02-16 00:00:00,customer17850@mail.com,L,37.0,Jakarta,Gold,2313.0,1.0
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,NaN,...,YES,2011-02-13 00:00:00,2011-02-16 00:00:00,customer17850@mail.com,L,37.0,Jakarta,Gold,2313.0,1.0


In [11]:
df_master_analisis.info()

<class 'pandas.DataFrame'>
RangeIndex: 78035 entries, 0 to 78034
Data columns (total 25 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   InvoiceNo                 78035 non-null  str           
 1   StockCode                 78035 non-null  str           
 2   Description               78035 non-null  str           
 3   Quantity                  78035 non-null  int64         
 4   InvoiceDate               78035 non-null  datetime64[us]
 5   UnitPrice                 78035 non-null  float64       
 6   CustomerID                78035 non-null  int64         
 7   Country                   78035 non-null  str           
 8   total_amount              78035 non-null  float64       
 9   coupon_code               4047 non-null   str           
 10  coupon_discount           4047 non-null   float64       
 11  status                    78035 non-null  str           
 12  customer_id               780

### Hasil untuk Data Owner

In [12]:
kupon = df_master_analisis[df_master_analisis['coupon_code'].notna()]
inv = kupon.drop_duplicates(subset='InvoiceNo')

print('Total transaksi pakai kupon :', len(inv), 'invoice')
print('Total nilai transaksi       :', round(kupon['total_amount'].sum(), 2))
print()
print('Profil loyalitas pemakai kupon:')
print(inv['membership_tier'].value_counts())
print()
print('Status pengiriman:')
print(inv['status'].value_counts())
print()
inv_kirim = inv.dropna(subset=['status'])
terkirim = (inv_kirim['status'] == 'Delivered').sum()
print(f'Yang sudah terkirim: {terkirim} dari {len(inv_kirim)} ({terkirim/len(inv_kirim)*100:.1f}%)')

Total transaksi pakai kupon : 203 invoice
Total nilai transaksi       : 68949.57

Profil loyalitas pemakai kupon:
membership_tier
Silver      51
Bronze      49
Gold        46
Platinum    43
Name: count, dtype: int64

Status pengiriman:
status
Delivered     153
In Transit     25
Shipped        21
Cancelled       4
Name: count, dtype: int64

Yang sudah terkirim: 153 dari 203 (75.4%)


## Bagian C.1 - Kamus Data

In [13]:
def generate_data_dictionary(df):
    kamus = pd.DataFrame({
        'nama_kolom': df.columns,
        'tipe_data_sistem': [str(t) for t in df.dtypes],
        'jumlah_missing': df.isna().sum().values,
        'persen_missing': (df.isna().mean().values * 100).round(2),
        'jumlah_nilai_unik': df.nunique(dropna=True).values,
    })

    sampel = []
    for col in df.columns:
        nilai = df[col].dropna().unique()[:3]
        sampel.append(', '.join(str(v) for v in nilai))
    kamus['sampel_nilai'] = sampel

    return kamus


kamus = generate_data_dictionary(df_master_analisis)
kamus.to_csv('kamus_data_hasil.csv', index=False)
kamus

,nama_kolom,tipe_data_sistem,jumlah_missing,persen_missing,jumlah_nilai_unik,sampel_nilai
0,InvoiceNo,str,0,0.00,4005,"536365, 536366, 536367"
1,StockCode,str,0,0.00,2832,"85123A, 71053, 84406B"
2,Description,str,0,0.00,2860,"WHITE HANGING HEART T-LIGHT HOLDER, WHITE META..."
3,Quantity,int64,0,0.00,154,"6, 8, 2"
4,InvoiceDate,datetime64[us],0,0.00,3796,"2010-12-01 08:26:00, 2010-12-01 08:28:00, 2010..."
5,UnitPrice,float64,0,0.00,188,"2.55, 3.39, 2.75"
6,CustomerID,int64,0,0.00,1904,"17850, 13047, 12583"
7,Country,str,0,0.00,30,"United Kingdom, France, Australia"
8,total_amount,float64,0,0.00,1229,"15.3, 20.34, 22.0"
9,coupon_code,str,73988,94.81,4,"NRMFLASH15, AKHIRBULAN25, DISKON20"


## Bagian C.2 - Validasi Kualitas Data

In [14]:
# 1. tidak boleh ada duplikat di primary key
duplikat = df_master_analisis.duplicated(subset=['InvoiceNo', 'StockCode']).sum()
assert duplikat == 0, f'ada {duplikat} baris duplikat'
print('cek 1 lolos - tidak ada duplikat primary key')

# 2. total_amount tidak boleh negatif
negatif = (df_master_analisis['total_amount'] < 0).sum()
assert negatif == 0, f'ada {negatif} nilai negatif'
print('cek 2 lolos - tidak ada total_amount negatif')

# 3. tanggal pesanan tidak boleh melebihi hari ini
batas = pd.Timestamp.now()
future = (df_master_analisis['InvoiceDate'] > batas).sum()
assert future == 0, f'ada {future} tanggal di masa depan'
print('cek 3 lolos - tidak ada future date')

print()
print('semua validasi lolos, data siap dipakai')

cek 1 lolos - tidak ada duplikat primary key
cek 2 lolos - tidak ada total_amount negatif
cek 3 lolos - tidak ada future date

semua validasi lolos, data siap dipakai
